# Why Did Claude Pick Those Tickers?

In the [Session 1 interview](../session-1/interview.md), each archetype maps to a curated list of 20 to 24 tickers. The lists encode two priors at once: the investor's stated objective (income, balanced, growth) and a sector-balance constraint that no archetype puts all its weight on a single industry. The lists were not learned from data; Claude wrote them.

This notebook asks one question: __could a learning-based ticker picker have done better?__ We benchmark the curated archetype baskets against a per-sector sparse bandit on the real 2025-2026 hold-out window, with the bandit trained on 2014-2024 history. The bandit gets the same sector-balance constraint the archetypes do; the only thing it adds is data.

> __The narrative arc:__
>
> The Session 3 [Ticker-Picker Bandit](../session-3/eCornell-AI-Finance-S3-Example-Core-TickerPickerBandit-May-2026.ipynb) ran one bandit over $2^{22} - 1$ subsets of a 22-ticker universe. At production scale, with $K \approx 413$ tickers, the action space explodes to $\binom{K}{K_{\text{basket}}} \approx 10^{27}$. We split the problem into 11 small per-sector bandits and let each pick its own slots. The result: at a basket size that matches the archetype lists, the bandit beats four of the five curated baskets at the median seed and ties the only one it cannot reliably beat.

The learning objectives below frame the theory recap and the two results tables that follow.

> __Learning Objectives:__
>
> By the end of this notebook, we will be able to:
> * __Reformulate basket selection as 11 small per-sector bandits:__ State the action, the arm, and the reward for a single sector and explain why the per-sector decomposition collapses the action space. Recognize that the cross-sectional reward strips away the market-beta variance that dominates raw forward returns.
> * __Read a Monte Carlo distribution against a deterministic baseline:__ Compare the trained bandit's median-seed outcome to the deterministic Claude-curated archetype baskets and the Full-Universe baseline. Distinguish a single-trial number from the distribution it was drawn from.
> * __Diagnose when picking adds value vs when diversification does:__ Identify the basket size at which per-sector picking becomes a real edge, and recognize why a curated archetype that aligns with the realized regime is the hardest baseline to beat.

Let's walk through the theory and read the bake-off.

___

## Section 1: Theory Recap
The picker we benchmark in this notebook reuses the [combinatorial epsilon-greedy bandit from Session 3](../session-3/eCornell-AI-Finance-S3-Example-Core-TickerPickerBandit-May-2026.ipynb) but reorganizes it. The Session 3 setup picked a basket from one universe; the Session 4 production setup picks 11 baskets, one per GICS sector, each with its own arms and reward.

> __Why the per-sector reformulation:__
>
> The Session 3 bandit enumerated $2^{22} - 1 \approx 4.2$ million arms by storing one arm-mean per subset. At production scale ($K = 413$), the analogous table $\binom{K}{16} \approx 10^{27}$ cannot be visited or stored. The per-sector decomposition replaces it with $\sum_{s=1}^{S} \binom{N_s}{q_s} \approx 7000$ arms total, five orders of magnitude smaller, by letting each sector's bandit handle its own slots.

The setup formalizes one bandit per sector with a quota that fixes how many slots that sector contributes to the assembled basket.

> __Per-sector setup:__
>
> * Universe: $K = 413$ S&P 500 tickers, partitioned into $S = 11$ GICS sectors with sector-$s$ size $N_s$.
> * Quota vector: $\mathbf{q} = (q_1, \ldots, q_S)$ with $\sum_s q_s = K_{\text{basket}}$. The bandit picks exactly $q_s$ tickers from sector $s$.
> * Arms: a sector-$s$ arm is a sorted $q_s$-tuple of within-sector ticker indices. Sector $s$ has $\binom{N_s}{q_s}$ arms.
> * Reward (sector-relative): for arm $\mathcal{A}_s$ on training day $t$ over horizon $h = 21$ trading days, the reward is given by:

$$R_s(\mathcal{A}_s, t) \;=\; r_{\mathcal{A}_s}(t, t+h) \;-\; r_{\text{EW}, s}(t, t+h)$$

where $r_{\mathcal{A}_s}$ is the basket's Cobb-Douglas-allocated buy-and-hold log return and $r_{\text{EW}, s}$ is the equal-dollar-weighted log return of all tickers in sector $s$ over the same window.

The sector-relative reward is the key design choice. Raw forward returns are dominated by market beta common to every basket; the residual cross-sectional dispersion is exactly the picking signal we want to learn. Subtracting the equal-weighted sector benchmark strips that common variance out and leaves the bandit a high-signal target.

The pseudocode below specifies one sector's bandit. We run $S = 11$ such bandits in parallel, one per GICS sector, and concatenate the winning sub-baskets into a single deployable basket of size $K_{\text{basket}} = \sum_s q_s$.

### Algorithm: Per-Sector Sparse Bandit (Offline Training)

__Initialize__: Given a sector $s$ with within-sector ticker indices $\mathcal{N}_s = \{1, \ldots, N_s\}$, a quota $q_s \in \mathbb{Z}_{>0}$, the action set $\mathcal{A}_s = \{\mathcal{B} \subseteq \mathcal{N}_s : |\mathcal{B}| = q_s\}$ of cardinality $|\mathcal{A}_s| = \binom{N_s}{q_s}$, a training-day set $\mathcal{D}_{\text{train}} \subset \mathbb{Z}$, a forward horizon $h$ (trading days), an iteration budget $T$, and an exploration floor $\varepsilon_{\min} \in (0, 1)$, set $\hat{\mu}_{\mathcal{B}} \gets 0$ and $n_{\mathcal{B}} \gets 0$ for each visited arm $\mathcal{B} \in \mathcal{A}_s$.

For $t = 1, \ldots, T$ __do__:

1. Compute the exploration rate: $\varepsilon_t \gets \max(\varepsilon_{\min},\; t^{-1/3} \cdot (|\mathcal{A}_s| \ln t)^{1/3})$.
2. Draw $p \sim \mathrm{Uniform}[0, 1]$ and select an arm:
    - If $p \leq \varepsilon_t$ or no arm has been visited yet, *explore*: sample $\mathcal{B}_t$ uniformly from $\mathcal{A}_s$.
    - Otherwise, *exploit*: set $\mathcal{B}_t \gets \arg\max_{\mathcal{B} \,\text{visited}} \hat{\mu}_{\mathcal{B}}$.
3. Sample a training day $d_t \sim \mathrm{Uniform}(\mathcal{D}_{\text{train}})$ and observe the sector-relative reward $r_t \gets R_s(\mathcal{B}_t, d_t) = r_{\mathcal{B}_t}(d_t, d_t + h) - r_{\text{EW}, s}(d_t, d_t + h)$.
4. Update the pulled arm:
    $$n_{\mathcal{B}_t} \gets n_{\mathcal{B}_t} + 1, \qquad \hat{\mu}_{\mathcal{B}_t} \gets \hat{\mu}_{\mathcal{B}_t} + \frac{1}{n_{\mathcal{B}_t}}\left(r_t - \hat{\mu}_{\mathcal{B}_t}\right)$$

__Output__: Return the learned sub-basket $\mathcal{B}_s^{\star} = \arg\max_{\mathcal{B} \,\text{visited}} \hat{\mu}_{\mathcal{B}}$, the visited-arm means $\{\hat{\mu}_{\mathcal{B}}\}$, pull counts $\{n_{\mathcal{B}}\}$, and the chosen-arm history $\{(\mathcal{B}_t, r_t)\}_{t=1}^{T}$.

The 11 sector winners assemble into the full deployable basket $\mathcal{B}^{\star} = \bigcup_{s=1}^{S} \mathcal{B}_s^{\star}$, which the daily Cobb-Douglas rebalancing engine walks through the 2025-2026 hold-out window.

The implementation lives in three scripts (linked here for reference; the notebook only loads their saved results below):

* [`scripts/bandit/per_sector_bandit.jl`](scripts/bandit/per_sector_bandit.jl) trains one bandit per sector, assembles the basket, deploys on the hold-out window, and saves a single-seed scorecard.
* [`scripts/bandit/monte_carlo_per_sector_bandit.jl`](scripts/bandit/monte_carlo_per_sector_bandit.jl) repeats the same training and deployment with 30 different seeds and saves the distribution of hold-out metrics.
* [`scripts/bandit/compare_archetypes.jl`](scripts/bandit/compare_archetypes.jl) deploys each Session 1 archetype basket on the same hold-out window and joins the result with the bandit's MC distribution and the Full-Universe baseline.

___

## Section 2: Results
We split the timeline into a training window and a hold-out window. The bandit sees only training-window data when it learns; every metric we report below is computed on the hold-out window.

> __Data windows:__
>
> * __Training:__ 2014-01-03 to 2024-12-31, ~10 years of daily closes for the 413-ticker universe. The bandit samples training days from this window and computes a 21-day forward sector-relative reward for each candidate arm.
> * __Hold-out:__ 2025-01-02 to 2026-04-22, 326 trading days of daily closes. Every assembled basket (bandit, archetype, or Full-Universe) is forward-walked through the daily Cobb-Douglas rebalancing engine on this window with the same drawdown and turnover rules.
> * __Universe filter:__ S&P 500 tickers with full OHLC coverage on every training day and every hold-out day. The filter drops delisted names and ETFs; what remains is the 413-ticker production universe.

The bandit's exploration order depends on its random seed, so a single training run produces a noisy point estimate. We therefore train the picker 30 times with seeds $1001, \ldots, 1030$, deploy each resulting basket on the same hold-out window, and report the distribution of outcomes alongside the deterministic archetype and Full-Universe baselines.

The code block below loads the saved bake-off results. No training happens in this notebook.

In [ ]:
include("Include.jl")

results = load(joinpath(_ROOT, "scripts", "bandit", "compare_archetypes_results.jld2"))

@assert results["K_COMPARE"] == 22  "expected K_COMPARE = 22 (uniform q_s = 2 across 11 sectors)"
@assert results["MC_N_SEEDS"] == 30 "expected 30 Monte Carlo seeds"

println("Bake-off K_COMPARE  : ", results["K_COMPARE"], " (uniform q_s = 2 across all 11 sectors)")
println("Monte Carlo seeds   : ", results["MC_N_SEEDS"], " (seeds ", results["MC_SEED_BASE"]+1, ":", results["MC_SEED_BASE"]+results["MC_N_SEEDS"], ")")
println("Forward window      : ", results["forward_first_date"], " to ", results["forward_last_date"], " (", results["n_fwd"], " trading days)")

> __Monte Carlo distribution at $K_{\text{basket}} = 22$:__
>
> Each row below summarizes the 30-seed distribution of one strategy's hold-out metrics. The Sector-Bandit rows are the per-sector bandit trained on 2014-2024 with sector-relative reward; the Random-per-sector rows pick $q_s$ tickers uniformly at random within each sector (same sector quotas, no learning). Compare medians across rows, not single values within a row.

In [ ]:
let
    sb_W = results["sb_W_T_mc"];   rs_W = results["rs_W_T_mc"]
    sb_S = results["sb_sharpe_mc"]; rs_S = results["rs_sharpe_mc"]
    sb_D = results["sb_dd_mc"];    rs_D = results["rs_dd_mc"]
    qstats(x) = (round(minimum(x); digits=3), round(quantile(x, 0.25); digits=3),
                 round(median(x);   digits=3), round(quantile(x, 0.75); digits=3),
                 round(maximum(x); digits=3))
    rows = NamedTuple[]
    for (label, w, s, d) in [
            ("Sector-Bandit",     sb_W, sb_S, sb_D),
            ("Random-per-sector", rs_W, rs_S, rs_D)]
        wmin, w25, wmed, w75, wmax = qstats(w)
        smin, s25, smed, s75, smax = qstats(s)
        dmin, d25, dmed, d75, dmax = qstats(d)
        push!(rows, (Strategy = label, Metric = "W_T / W_0",
            Min = wmin, Q25 = w25, Median = wmed, Q75 = w75, Max = wmax))
        push!(rows, (Strategy = label, Metric = "Sharpe",
            Min = smin, Q25 = s25, Median = smed, Q75 = s75, Max = smax))
        push!(rows, (Strategy = label, Metric = "Max DD %",
            Min = dmin, Q25 = d25, Median = dmed, Q75 = d75, Max = dmax))
    end
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false)
end

The MC distribution above is one half of the comparison. The other half is the deterministic baselines: the 5 Claude-curated archetype baskets from the Session 1 interview, plus the Full-Universe basket of all 413 tickers. Each of those is a fixed list, so it produces a single number per metric.

> __Hold-out bake-off (single point per strategy):__
>
> The 5 archetypes and the Full-Universe baseline contribute one row each (no randomness). The Sector-Bandit and Random-per-sector rows report the median across the 30 Monte Carlo seeds. Sorted by hold-out Sharpe, descending.

In [ ]:
let
    arch  = results["archetype_metrics"]
    kept  = results["archetype_kept"]
    fu    = results["full_universe_metrics"]
    n_fwd = results["n_fwd"]

    sb_W_med = median(results["sb_W_T_mc"])
    sb_S_med = median(results["sb_sharpe_mc"])
    sb_D_med = median(results["sb_dd_mc"])
    rs_W_med = median(results["rs_W_T_mc"])
    rs_S_med = median(results["rs_sharpe_mc"])
    rs_D_med = median(results["rs_dd_mc"])

    function row(strategy::String, K::Int, w::Real, ann::Real, dd::Real, sh::Real)
        return (Strategy = strategy, K = K,
            W_T_over_W0 = round(w; digits = 3),
            Ann_ret_pct = round(ann; digits = 2),
            Max_DD_pct  = round(dd; digits = 1),
            Sharpe      = round(sh; digits = 3))
    end

    rows = NamedTuple[]
    for (name, m) in arch
        push!(rows, row(name, length(kept[name]),
            m.W_T_over_W0, m.ann_ret * 100, m.max_dd * 100, m.sharpe))
    end
    push!(rows, row("Sector-Bandit (MC median)", results["K_COMPARE"],
        sb_W_med, log(sb_W_med) * (252.0 / n_fwd) * 100, sb_D_med, sb_S_med))
    push!(rows, row("Random-per-sector (MC median)", results["K_COMPARE"],
        rs_W_med, log(rs_W_med) * (252.0 / n_fwd) * 100, rs_D_med, rs_S_med))
    push!(rows, row("Full-Universe", 413,
        fu.W_T_over_W0, fu.ann_ret * 100, fu.max_dd * 100, fu.sharpe))
    sort!(rows; by = r -> -r.Sharpe)
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false)
end

The bake-off table compares medians, but the bandit is a distribution. The cleaner question is: across all 30 trained bandits, what fraction of seeds beat each archetype on Sharpe?

> __Where does each archetype sit in the bandit's Sharpe distribution?__
>
> "Bandit beats it %" is the percentage of the 30 bandit seeds whose hold-out Sharpe exceeds the archetype's Sharpe. A value above 50% means a typical bandit run beats that archetype; a value near 50% is a coin flip.

In [ ]:
let
    arch = results["archetype_metrics"]
    sb_S = results["sb_sharpe_mc"]
    rows = NamedTuple[]
    for (name, m) in sort(collect(arch); by = p -> -p[2].sharpe)
        wins = sum(>(m.sharpe), sb_S)
        push!(rows, (
            Archetype           = name,
            Archetype_Sharpe    = round(m.sharpe; digits = 3),
            Bandit_beats_it_pct = round(100 * wins / length(sb_S); digits = 0),
            Archetype_beats_pct = round(100 * (length(sb_S) - wins) / length(sb_S); digits = 0)))
    end
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false)
end

The histogram below renders the same comparison visually: the 30-seed bandit Sharpe distribution at $K_{\text{basket}} = 22$, with vertical lines marking each archetype's deterministic Sharpe and the Full-Universe Sharpe. Archetypes to the left of the bandit median lose to a typical bandit run; archetypes to the right beat it.

In [ ]:
let
    sb_S = results["sb_sharpe_mc"]
    arch = results["archetype_metrics"]
    fu   = results["full_universe_metrics"]

    p = histogram(sb_S; bins = 10,
        bg = :gray95, framestyle = :box,
        xlabel = "Hold-out Sharpe",
        ylabel = "Bandit seeds (count out of 30)",
        title  = "Bandit Sharpe distribution at K_basket = 22 vs deterministic baselines",
        titlefontsize = 11, lw = 1.5,
        size = (980, 500), label = "Bandit seeds (30)",
        legend = :topleft,
        foreground_color_legend = nothing,
        legend_background_color = RGBA(1.0, 1.0, 1.0, 0.6),
        color = :steelblue, fillalpha = 0.55)
    vline!(p, [median(sb_S)]; lw = 2.5, color = :black, ls = :dash,
        label = "Bandit MC median")
    vline!(p, [fu.sharpe]; lw = 2.5, color = :red,
        label = "Full-Universe (K=413)")
    arch_colors = [:darkblue, :seagreen, :goldenrod, :purple, :orangered]
    for (i, (name, m)) in enumerate(sort(collect(arch); by = p -> -p[2].sharpe))
        vline!(p, [m.sharpe]; lw = 2.0,
            color = arch_colors[mod(i - 1, length(arch_colors)) + 1],
            label = name)
    end
    plot(p)
end

> __What this tells us:__
>
> Four of the five archetype baskets sit below the bandit's MC-median Sharpe at $K_{\text{basket}} = 22$. The bandit's MC-median Sharpe is statistically indistinguishable from the Full-Universe baseline despite holding only 22 of the 413 tickers; the per-sector picker is buying real concentration without paying a Sharpe cost.

The one archetype the bandit cannot reliably beat is __Aggressive Growth__. Its Sharpe sits near the upper tail of the bandit distribution, where roughly half of bandit seeds happen to find similarly aggressive tilts and the other half do not. Aggressive Growth wins because its tilt toward Information Technology and Consumer Discretionary aligned with the realized 2025-2026 hold-out window, and that alignment is *information the bandit cannot extract from training history alone*. So the curated basket beats a typical bandit run by encoding regime intent the bandit had to learn from data.

This is the answer to the title's question. Claude picked sector-balanced lists weighted by archetype intent because that combination, sector diversification plus aligned thematic tilt, is the strongest baseline a per-sector bandit has to beat. At basket sizes too small for uniform within-sector diversification (for example $K_{\text{basket}} = 16$ with $q_s = 1$ in 6 of 11 sectors), the bandit loses to the archetypes; at $K_{\text{basket}} = 22$ with uniform $q_s = 2$, it ties or wins on four of five. The picking edge appears only when the structure is wide enough to support it.

___

## Summary
The per-sector sparse bandit beats four of five Claude-curated archetype baskets at the median seed when the basket size matches the archetype range ($K_{\text{basket}} = 22$, uniform $q_s = 2$). At smaller basket sizes the bandit loses to the curated lists; at the matching size it ties Full-Universe Sharpe with a 5% basket. The only archetype it cannot reliably beat is the one whose thematic tilt matched the realized regime, which is information the bandit cannot extract from training history alone.

> __Key Takeaways:__
>
> * __Per-sector decomposition makes the action space tractable:__ Splitting one $\binom{K}{K_{\text{basket}}}$ bandit into 11 small per-sector bandits collapses an arm count of $\sim 10^{27}$ to $\sim 7000$. The cross-sectional reward strips market-beta variance and leaves the picker a high-signal target.
> * __Read distributions, not single trials:__ A single-seed bandit number is a noisy point estimate. The 30-seed median is the honest comparison against deterministic baselines; the bandit's MC-median Sharpe at $K_{\text{basket}} = 22$ ties Full-Universe and beats four of five archetypes.
> * __Curated intent is hard to beat when it aligns with the regime:__ The Aggressive Growth archetype's Tech-heavy tilt aligned with the 2025-2026 hold-out window, putting it just above the bandit's MC median. The bandit can match it on roughly half of seeds but not reliably outperform it.

This closes the question that opened the notebook. The Session 1 archetype lists are a strong baseline because they encode investment intent and sector balance; a per-sector bandit can match both at the right basket size, but neither is automatic when the basket is too small or the intent is unknown.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use real historical data and a frozen SIM calibration; conclusions about a single forward window do not generalize to other markets or time periods.

___